## Preparation

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import sys
sys.path.append('../')
import os
os.environ['R_HOME'] = "D:/anaconda3/envs/stacame/lib/R"
os.environ['R_USER'] = "D:/anaconda3/envs/stacame/lib/python3.11/site-packages/rpy2/"
import rpy2.robjects as robjects
import rpy2.robjects.numpy2ri

import STACAME
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import scipy.linalg
from scipy.sparse import csr_matrix

import pandas as pd

import torch
from STACAME.analysis import merge_embedding
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score as ari_score
import seaborn as sns
import colorcet as cc

In [3]:
def clustering_umap_spatial(adata_dict, key_umap='STAGATE'):
    k = 0
    for species_id, adata in adata_dict.items():
        if species_id == 'Zebrafish':
            adata.obs['annotation'] = adata.obs['layer_annotation']
        if k == 0:
            embedding_X = adata.obsm[key_umap]
            embedding_spatial = adata.obsm['spatial']
            embedding_obs_name = list(adata.obs_names)
            embedding_slice_name = list(adata.obs['slice_name']) 
            embedding_batch_name = list(adata.obs['batch_name'])
            embedding_species_id = list(adata.obs['species_id'])
            embedding_annotation = list(adata.obs['annotation']) 
        else:
            embedding_X = np.concatenate((embedding_X, adata.obsm[key_umap]), axis=0)
            embedding_spatial = np.concatenate((embedding_spatial, adata.obsm['spatial']), axis=0)
            embedding_obs_name = embedding_obs_name + list(adata.obs_names)
            embedding_slice_name = embedding_slice_name + list(adata.obs['slice_name']) 
            embedding_batch_name = embedding_batch_name + list(adata.obs['batch_name'])
            embedding_species_id = embedding_species_id + list(adata.obs['species_id'])
            embedding_annotation = embedding_annotation + list(adata.obs['annotation'])

        k += 1
        #Visualize UMAP of each species
        sc.pp.neighbors(adata,  n_neighbors=20, use_rep=key_umap, metric='cosine',  random_state=666)
        sc.tl.louvain(adata, random_state=666, key_added="louvain", resolution=0.5)
        sc.tl.umap(adata, min_dist=1, random_state=666)
        plt.rcParams['font.sans-serif'] = "Arial"
        plt.rcParams["figure.figsize"] = (3, 3)
        plt.rcParams['font.size'] = 10
        print(adata)
        print(adata.obs['annotation'].unique())
        num_clusters = len(adata.obs['annotation'].unique())
        STACAME.mclust_R(adata, num_cluster=num_clusters, used_obsm=key_umap)
        print('mclust, ARI = %01.3f' % ari_score(adata.obs['annotation'], adata.obs['mclust']))
        sc.pl.umap(adata, color=['batch_name', 'louvain', 'annotation', 'mclust'], ncols=3, wspace=0.7, show=True)

        adata_dict[species_id] = adata

    adata_embedding = ad.AnnData(X = embedding_X, obs=embedding_obs_name)
    adata_embedding.obsm['spatial'] = embedding_spatial
    adata_embedding.obs['slice_name'] = embedding_slice_name
    adata_embedding.obs['batch_name'] = embedding_batch_name
    adata_embedding.obs['species_id'] = embedding_species_id
    adata_embedding.obs['annotation'] = embedding_annotation
    
    return adata_dict, adata_embedding

## Process of datasets

In [4]:
root_data_path = './Data/1_DLPFC/'
Gene_map_raw_path = './Data/1_DLPFC/Macaque_Human.tsv'
rad_cutoff_dict = {'Macaque':210, 'Human':201}
species_section_ids = {'Macaque':['macaque_T127_DLPFC'],
                       'Human':['human_151673_modified']}
species_ortholog_column_dict = {'Macaque':'Gene name', 
                                'Human':'Human gene name'}
species_ortholog_type_dict = {'Human':'Human homology type'}
species_id_map = {'Macaque':0, 'Human':1}


STACAME_processer = STACAME.STACAME_processer(root_data_path=root_data_path,
                 Gene_map_raw_path=Gene_map_raw_path, 
                 species_section_ids = species_section_ids, 
                 species_ortholog_column_dict = species_ortholog_column_dict, 
                 species_ortholog_type_dict = species_ortholog_type_dict, 
                 species_id_map = species_id_map, 
                 rad_cutoff_dict = rad_cutoff_dict,
                 gene_cap_upper_dict = {'Macaque':'upper', 'Human':'upper'},
                 Down_sampling_adata = None, 
                 n_top_genes = 300, 
                 homo_n_top_genes = 5000, 
                 cross_species_neibors_K_mnn = 150,
                 total_normalize = {'Macaque':2e4, 'Human':2e4},
                 min_cells = 20, 
                 if_hvg_before_mnn = False, 
                 if_combat_mnn = False, 
                 if_pca_before_mnn = True, 
                 pca_dim_before_mnn = 50, 
                 if_return_concat_adata = True)
adata_dict, triplet_ind_species_dict, edge_ndarray_species, adata_whole = STACAME_processer.load_process_adata()

print(adata_whole)
sc.pp.highly_variable_genes(adata_whole, flavor='seurat_v3', n_top_genes=3000)
sc.tl.pca(adata_whole, svd_solver='arpack', n_comps=50)
adata_whole = adata_whole[:, adata_whole.var.highly_variable].copy()
adata_whole.X = scipy.sparse.csr_matrix(adata_whole.X)

self.rad_cutoff_dict: {'Macaque': {'macaque_T127_DLPFC': 210}, 'Human': {'human_151673_modified': 201}}
--------------------------Species-Macaque-------------------------------
Species: Macaque Section: macaque_T127_DLPFC
(18375, 2)
Before flitering:  (18375, 15096)
After flitering:  (18375, 12471)
Number of genes: 12471
Before flitering:  (18375, 15096)
After flitering:  (18375, 12471)
Number of hvgs: 5000
Number of common hvgs: 5000
--------------------------Species-Human-------------------------------
Species: Human Section: human_151673_modified
(3611, 2)
Before flitering:  (3611, 18002)
After flitering:  (3611, 15112)
Number of genes: 15112
Before flitering:  (3611, 18002)
After flitering:  (3611, 15112)
Number of hvgs: 5000
Number of common hvgs: 5000
Normalizing data and get spatial neigbors...
--------------------------Species-Macaque-------------------------------
---------Section-macaque_T127_DLPFC---------
------Calculating spatial graph...
Using Euclidean distance for spati

## Running STACAME

In [ ]:
%%time
import importlib
from STACAME.train_STACAME import train_STACAME_subgraph_GAN
used_device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
pretrain_device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
for k,v in adata_dict.items():
    print(k, v)

# loss weight
device_ids = [1]    
for spe, adata in adata_dict.items():
    print(spe, adata)
adata_species_dict = train_STACAME_subgraph_GAN(adata_dict,
                           triplet_ind_species_dict, 
                           edge_ndarray_species, 
                           hidden_dims=[256, 20], 
                           verbose=True, 
                           knn_neigh = 20,  
                           key_added = 'STACAME',
                           device=used_device, 
                           pretrain_device = pretrain_device,
                           stagate_epoch={'Macaque':100, 'Human':100}, 
                           n_epochs_species = 1500,
                           margin_species=5,
                           lr=0.001, 
                           lr_species = 0.001,
                           beta=1,
                           mse_beta = 1, 
                           tri_beta = 5,
                           mmd_beta = 2,
                           gan_beta = 5, 
                           gan_epoch = 3,
                            concate_pca_dim = 128,
                            mmd_batch_size = 1024,
                            if_knn_mnn_graph = True,
                            if_batch_pretrain = True,
                            batch_size_dict = {'Macaque': 2048, 'Human':2048},
                            batch_size = 1024, 
                            adata_whole = adata_whole)

Macaque AnnData object with n_obs × n_vars = 18375 × 10999
    obs: 'gene_area', 'origin_name', 'region_full_name', 'region_acronym_name', 'annotation', 'region_name', 'gene_area_origin', 'species_id', 'slice_name', 'batch_name'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg', 'edgeList', 'homo_highly_variable', 'highly_variable'
    obsm: 'spatial'
Human AnnData object with n_obs × n_vars = 3611 × 11790
    obs: 'in_tissue', 'array_row', 'array_col', 'Ground Truth', 'annotation', 'region_name', 'gene_area', 'species_id', 'slice_name', 'batch_name'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg', 'edgeList', 'homo_highly_variable', 'highly_variable'
    obsm: 'spatial'
Macaque AnnData object with n_obs × n_vars = 18375 × 10999
    obs: 'gene_area', 'origin_name', 'region_full_name', 'region_acronym_name', 'annotation', 'region_name', 'gene_area_origin', 'species_id', 'sli

  1%|██▏                                                                                                                                                                                                                             | 1/100 [00:03<06:11,  3.76s/it]

Epoch = 0, mse loss = nan


  2%|████▍                                                                                                                                                                                                                           | 2/100 [00:07<05:44,  3.51s/it]

Epoch = 1, mse loss = nan


...

  0%|▏                                                                                                                                                                                                                              | 1/1500 [00:01<29:41,  1.19s/it]

---------------------------------Epoch 0-----------------------------------
MSE loss:16.29153060913086,  Cross species triplets:7.794182777404785, MMD loss:5.846634387969971, GAN loss: -0.02294776774942875
Cosine cross species loss:0.10740135610103607


  7%|██████████████▉                                                                                                                                                                                                              | 101/1500 [00:53<11:26,  2.04it/s]

---------------------------------Epoch 100-----------------------------------
MSE loss:3.182220220565796,  Cross species triplets:2.5909423828125, MMD loss:0.42950010299682617, GAN loss: -0.13610759377479553
Cosine cross species loss:0.1378621906042099


 10%|██████████████████████▌                                                                                                                                                                                                      | 153/1500 [01:21<14:02,  1.60it/s]

## Post process of STACAME embeddings

In [ ]:
adata_dict, adata_embedding = clustering_umap_spatial(adata_dict, key_umap='STACAME')

## Save embeddings

In [ ]:
output_path = root_data_path + 'output_STACAME_minibatch/'
if not os.path.exists(output_path):
    os.makedirs(output_path)
for species_id, adata in adata_dict.items():
    print(adata.obsm['STACAME'].shape)
    if 'edgeList' in adata.uns.keys():
        del adata.uns['edgeList']
    adata.write(output_path + f'{species_id}.h5ad')
    adata_temp = adata[:, adata.uns['highly_variable']]
    adata_temp.write(output_path + f'adata_{species_id}_expression.h5ad')


from STACAME.analysis import merge_embedding
adata_embedding = merge_embedding(adata_dict, key_umap = 'STACAME')
adata_embedding.obs['region_name'] = adata_embedding.obs['annotation']
adata_embedding.write(output_path + 'adata_embedding.h5ad')

In [ ]:
adata_embedding = sc.read_h5ad(output_path + 'adata_embedding.h5ad')

## Visualization of seperate mclust clusters and ARI

In [ ]:
from sklearn.metrics import adjusted_rand_score as ari_score

Batch_list = []
for species_id in adata_dict.keys():
    adata = adata_dict[species_id]
    Batch_list.append(adata)

species_list = list(adata_dict.keys())

import matplotlib.pyplot as plt
spot_size = 100
title_size = 12
ARI_list = []
for bb in range(2):
    ARI_list.append(round(ari_score(Batch_list[bb].obs['annotation'], Batch_list[bb].obs['mclust']), 3))

fig, ax = plt.subplots(2, 2, figsize=(14, 8), gridspec_kw={'wspace': 0.5, 'hspace': 0.4})
spot_size = 120
_sc_0 = sc.pl.spatial(Batch_list[0], img_key=None, color=['mclust'], title=[''],
                      legend_loc='right margin', legend_fontsize=12, show=False, ax=ax[0][0], frameon=False,
                      spot_size=spot_size)
_sc_0[0].set_title("ARI=" + str(ARI_list[0]), size=title_size)
_sc_1 = sc.pl.spatial(Batch_list[0], img_key=None, color=['annotation'], title=[species_list[0] + ' Annotation'],
                      legend_loc='right margin', legend_fontsize=12, show=False, ax=ax[0][1], frameon=False,
                      spot_size=spot_size)

spot_size = 150
_sc_2 = sc.pl.spatial(Batch_list[1], img_key=None, color=['mclust'], title=[''],
                      legend_loc='right margin', legend_fontsize=12, show=False, ax=ax[1][0], frameon=False,
                      spot_size=spot_size)
_sc_2[0].set_title("ARI=" + str(ARI_list[1]), size=title_size)
_sc_3 = sc.pl.spatial(Batch_list[1], img_key=None, color=['annotation'], title=[species_list[1] + ' Annotation'],
                      legend_loc='right margin', legend_fontsize=12, show=False, ax=ax[1][1], frameon=False,
                      spot_size=spot_size)
plt.show()


## Visualization of shared mclust clusters and ARI

In [ ]:
plt.rcParams['font.sans-serif'] = "Arial"
plt.rcParams['font.size'] = 10
fig_format = 'png'
fig_dpi = 500
annotation_num = 7

fig_save_path = output_path

num_clusters =annotation_num
print(f'Mclust {num_clusters} clusters...')
STACAME.mclust_R(adata_embedding, num_cluster=num_clusters, used_obsm='STACAME')
Batch_list = []

for species_id in species_section_ids.keys():
    adata_temp = adata_embedding[adata_embedding.obs['species_id'].isin([species_id])]
    Batch_list.append(adata_temp)

species_list = list(species_section_ids.keys())
spot_size = 120
title_size = 10
ARI_list = []
for bb in range(2):
    ARI_list.append(round(ari_score(Batch_list[bb].obs['annotation'], Batch_list[bb].obs['mclust']), 2))

fig, ax = plt.subplots(2, 2, figsize=(10, 8), gridspec_kw={'wspace': 0.3, 'hspace': 0.1})

clust_list = list(set(list(Batch_list[0].obs['mclust'].unique()) + list(Batch_list[1].obs['mclust'].unique())))
color_list = sns.color_palette(cc.glasbey, n_colors=len(clust_list))
clust_palette = {k:v for k,v in zip(clust_list, color_list)}
palette = {k:clust_palette[k] for k in Batch_list[0].obs['mclust'].unique()}
_sc_0 = sc.pl.spatial(Batch_list[0], img_key=None, color=['mclust'], title=['mclust'],
                      legend_loc='right margin', legend_fontsize=10, show=False, ax=ax[0][0], frameon=False,
                      spot_size=spot_size, palette=palette)
_sc_0[0].set_title("ARI=" + str(ARI_list[0]), size=title_size)
color_list = sns.color_palette(cc.glasbey, n_colors=len(Batch_list[0].obs['annotation'].unique()))
region_list =  ['Layer 1', 'Layer 2', 'Layer 3', 'Layer 4', 'Layer 5','Layer 6']
color_list = sns.color_palette(cc.glasbey, n_colors=len(region_list))
palette = {k:v for k,v in zip([x for x in region_list], color_list)}
_sc_1 = sc.pl.spatial(Batch_list[0], img_key=None, color=['annotation'], title=[species_list[0] + ' annotation'],
                      legend_loc='right margin', legend_fontsize=10, show=False, ax=ax[0][1], frameon=False,
                      spot_size=spot_size, palette=palette)
palette = {k:clust_palette[k] for k in Batch_list[1].obs['mclust'].unique()}
spot_size = 150
_sc_2 = sc.pl.spatial(Batch_list[1], img_key=None, color=['mclust'], title=['mclust'],
                      legend_loc='right margin', legend_fontsize=10, show=False, ax=ax[1][0], frameon=False,
                      spot_size=spot_size, palette=palette)
_sc_2[0].set_title("ARI=" + str(ARI_list[1]), size=title_size)
color_list = sns.color_palette(cc.glasbey, n_colors=len(Batch_list[1].obs['annotation'].unique()))
region_list = ['Layer 1', 'Layer 2', 'Layer 3', 'Layer 4', 'Layer 5','Layer 6', 'WM']
color_list = sns.color_palette(cc.glasbey, n_colors=len(region_list))
palette = {k:v for k,v in zip([x for x in region_list], color_list)}
_sc_3 = sc.pl.spatial(Batch_list[1], img_key=None, color=['annotation'], title=[species_list[1] + ' annotation'],
                      legend_loc='right margin', legend_fontsize=10, show=False, ax=ax[1][1], frameon=False, 
                      spot_size=spot_size, palette=palette)

if not os.path.exists(fig_save_path):
    os.makedirs(fig_save_path)
plt.savefig(fig_save_path + 'common_mclust.png', format=fig_format, dpi=fig_dpi)
plt.show()

## UMAP of embeddings

In [ ]:
from STACAME.analysis import get_alignment_score, convert_dict2adata
from matplotlib import rcParams
import matplotlib.pyplot as plt
import seaborn as sns


sns.set(style='white')
TINY_SIZE = 11 # 39
SMALL_SIZE = 11  # 42
MEDIUM_SIZE = 12  # 46
BIGGER_SIZE = 12  # 46

umap_neighbor = 30
umap_size_dict = {'Macaque':3, 'Human':6}
fig_format = 'jpg'
macaque_color = '#CCAE3D' 
human_color = '#FE1613'
save_path = root_data_path + 'output_STACAME_minibatch/figs/'
if not os.path.exists(save_path):
    os.makedirs(save_path)

fig_dpi = 400
plt.rc('axes', labelsize=MEDIUM_SIZE)  # fontsize of the x and y labelsc
plt.rc('xtick', labelsize=TINY_SIZE)  # fontsize of the tick labels
plt.rc('ytick', labelsize=TINY_SIZE)  # fontsize of the tick labels
plt.rc('legend', fontsize=MEDIUM_SIZE)  # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial']

palette = {'Macaque':macaque_color, 'Human':human_color}
adata_embedding.obs['dataset'] = adata_embedding.obs['species_id']

sc.tl.pca(adata_embedding, svd_solver='arpack', n_comps=10)
sc.pp.neighbors(adata_embedding, n_neighbors=umap_neighbor, metric='cosine',
                use_rep='X')
sc.tl.umap(adata_embedding)

adata_macaque_embedding = adata_embedding[adata_embedding.obs['dataset'].isin(['Macaque'])]
adata_human_embedding = adata_embedding[adata_embedding.obs['dataset'].isin(['Human'])]
adata_umap_size_list = [umap_size_dict[x] for x in adata_embedding.obs['dataset']]

with plt.rc_context({"figure.figsize": (3, 2.5), "figure.dpi": (fig_dpi)}):
    sc.pl.umap(adata_macaque_embedding, color=['region_name'], return_fig=True, legend_loc='right margin', size=3).savefig(
        save_path + 'umap_macaque.' + fig_format, format=fig_format)

sc.pp.neighbors(adata_human_embedding, n_neighbors=umap_neighbor, metric='cosine', use_rep='X')
sc.tl.umap(adata_human_embedding)

with plt.rc_context({"figure.figsize": (3, 2.5), "figure.dpi": (fig_dpi)}):
    sc.pl.umap(adata_human_embedding, color=['region_name'], return_fig=True, size=3,
               legend_loc='right margin').savefig(
        save_path + 'umap_human.' + fig_format, format=fig_format)

rcParams["figure.subplot.left"] = 0.2
rcParams["figure.subplot.right"] = 0.9


rcParams["figure.subplot.left"] = 0.1
rcParams["figure.subplot.right"] = 0.68#0.6545
with plt.rc_context({"figure.figsize": (4, 3), "figure.dpi": (fig_dpi)}):
    fg = sc.pl.umap(adata_embedding, color=['dataset'], return_fig=True, legend_loc='right margin', title='', size= adata_umap_size_list, palette=palette)
    plt.title('')
    fg.savefig(save_path + 'umap_dataset_after_integration_rightmargin.' + fig_format, format=fig_format)


rcParams["figure.subplot.right"] = 0.9
with plt.rc_context({"figure.figsize": (3, 3), "figure.dpi": (fig_dpi)}):
    fg = sc.pl.umap(adata_embedding, color=['dataset'], return_fig=True, legend_loc=None, title='', size= adata_umap_size_list, palette=palette)
    plt.title('')
    fg.savefig(save_path + 'umap_dataset_after_integration.' + fig_format, format=fig_format)